# The heat-equation PINN loss

Write the loss of a physics-informed neural network for the heat equation $u_t = \alpha\, u_{xx}$ on $[0, 1] \times [0, 1]$ with initial condition $u(0, x) = \sin(\pi x)$ given as data and boundary condition $u(t, 0) = u(t, 1) = 0$ (lesson u21-l01).

```python
def heat_pinn_loss(model, tx_pde, tx_ic, u_ic, tx_bc, alpha):
    # Your code
    return total, l_pde, l_ic, l_bc
```

- `model`: an `nn.Module` mapping a float tensor of shape `(N, 2)` with columns $(t, x)$ to shape `(N, 1)`. Rows do not interact.
- `tx_pde`: shape `(N_r, 2)`, collocation points. It does **not** require grad.
- `tx_ic`: shape `(N_0, 2)`, initial points (their $t$ column is 0); `u_ic`: shape `(N_0, 1)`, the target values there.
- `tx_bc`: shape `(N_b, 2)`, boundary points; the target there is 0.
- `alpha`: a Python float.

Return four 0-dimensional tensors:

$$l_{\text{pde}} = \operatorname{mean}_i \big(u_t - \alpha u_{xx}\big)^2 \text{ at the rows of } \texttt{tx\_pde}, \quad l_{\text{ic}} = \operatorname{mean}_j \big(u(\texttt{tx\_ic}_j) - \texttt{u\_ic}_j\big)^2, \quad l_{\text{bc}} = \operatorname{mean}_k u(\texttt{tx\_bc}_k)^2,$$

and `total = l_pde + l_ic + l_bc`. The derivatives are those of `model` with respect to its inputs. `total.backward()` must give the correct gradients for every parameter of `model`, including the part that flows through $u_t$ and $u_{xx}$. Do not modify the inputs, and do not turn on `requires_grad` for the caller's tensors.

**Example.** For a model that computes $u = t^2 + x^3$ (no parameters needed), $u_t = 2t$ and $u_{xx} = 6x$. With `tx_pde = [[0.5, 1.0]]` and `alpha = 0.1` the residual is $1 - 0.6 = 0.4$, so `l_pde` is `0.16`.

**Restrictions.** Only `import torch` and `import math`. Use `torch.autograd.grad`; `torch.autograd.functional.jacobian`, `hessian` and the `torch.func` transforms (`vmap`, `jacrev`, `jacfwd`) are not allowed.

**Constraints.** $N_r \le 5000$, $N_0, N_b \le 500$. Time limit 1 second per call on CPU.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem1_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem1_submission.py
import math
import torch


def heat_pinn_loss(model, tx_pde, tx_ic, u_ic, tx_bc, alpha):
    # Your code
    return total, l_pde, l_ic, l_bc


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem1_submission as submission
importlib.reload(submission)
# e.g. submission.heat_pinn_loss(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9XAtz28iR/itz2qoYUCiYpB+5cFeb83q1ia/8Ktup5EpSsENiSGIFAggGsKhV9N+vv54ZYEBCsuTNnbdKBAYz3T397sZgrw/S5GAmDprp5KgcTw5G4mCjaklD1/tP6rTOFMY+rZVYK1kfqX82sk6LXLx/9fatyAqtMS9TWhe5W5xZsGne1ErT6OQZ3Sbpcpkumqy+opEnNKDyusL1AQDHZZrnsYO3KZImU3EuN4y9rIp5pjaTWDfzTao14ccsmWXFpUridFMWVQ1Epwd1US3WDELW64NzupjLPFcJP/xFLop5Knnxmki2l583ssQvPa7UZ3u1vEzcciKrKAF+KTOtwBal67hON6po6hgPpmOMypWhAVuJsJW4VtWGN8RkRbKpi1Ulkxh/MMwzefuOr4xTF0214J0XVbpKc5nxZFnVUxqsq0bd0P0yzZi71wdrWeW0nahkdp7xf39RWakqLeq1rIVloABeDBqOieDnZVVshF3uRg9/DqOz/Cx/YaeLVItFkdckr8dNntb68Xf4+f6xhUr3ZZp8/9jMJiIi8aoWiVqmBFWAV2LZ5AvsTguINOHB+NDQVssLRVfqLO+kK4wCiJ9p6OdInPz9xZv3r08+iizVNbakDFgA00Kvi8tc0KIPTS6Cz6lOiarwW/ER4GpRNYRWfVbVFS+irf0k06ypaGUlU60EbsVlWq+FFARPy5USl1Va04bFsqjEDyqfCS2vxCXIvZSanhb5aiRyQBVJsynFOk0Smp7mZVNrcM9IIbcsXRTlVXsDxQF/7W1O668EQc1LjH4jXhaZnIuX7/+qhayU0FDyCrzKQcojLd5MxfuqGAldkAEuZAaIab7SYtMQU+YKTCUWFbC3lB5nKfgwp/2vSZRLuSBtjIBpGo0hXClWDeQfGMyaYGXqqF5XSiZCl4oERrOqolmtsyvSlmwpiBaiYlFUYDQZDTG9FukSBNItm6PdhzUTMOXj+5OTH+OfXrz89O6DOAZ2bPgsX2SSsEMMwcl2oUpoSjg7ywX9K+mRmUUKRRxXSUC6mIzERq/cHMKbF2ByntgR/OuEG2BuByTOy2Dr1pIL8tY4CcFavdEl7T/NdS3zhQq2I/M8+qRyXVSht5zRqrqpcrGNEvKqi3UQRouyob8s5iA0kxVvU7xidCdVVVQeFLNlD1ZeRlLLqpJXRHe3D72WpQpWRT0ipczxl/Tz+OyANJu87NmBo0wTr+umzFSAndP8MOKllhbmKc1xkwArHInl2cE1AN6QwLXBJa41+R21LdWiJp249ubfELqOsgU5v7soG4mqLrLjiTp6PhLSXv6no5cs65JIbomVur6ijS6zQoIwjDPO/gNvMyuzP2zp0lzesh2zATs9HNqbe8TbaxGQQFJN7i2taTXRkWVBKMhTQAv9Z5f2mY8enlSm7AlzrEnzZQvcKjLknWWWicQMyy78sezCH1/xVFURx5gRIG4jt/iRcx2sxJG4DMMQsGmj6a9KKApiYgzjGzCVPp/Y0YnPMiP/IIJMViu4XQWFFdf0M4uerG7CnuwRyLJ7aGXfqMxUEJ0nrOmjkDlKRMi6ruzzswMWRwdkQF08Benm/J+rha8aTH9s+UDE7RmTz9UWAqQy26WYlAHU8uY7GOSMr+nRf1Q90jCLhnrCSCkFuXJOAlf6VnkwQk8i3ZI9+jFKrLCAwyjmRC2Oe+SsyP1BchbrIzfpkYPapzTWuSwpktf/v645K3L1QKdMKJ1WbqGSHNsPQgqJCVtvn54gSRfESCQuoU9Pf3M96gAwSJQqjz9RpucpmCXwE7Fvl7x9AGEvhGAoAkx+tu0xnsQSyJGYfz3f5b34bqYYu2B8X8t2CbYzHN13BRaRNEh0MA+HfA0t7/zM4PLWT8ztJcu2b9nYwch4O9Ls3JeVg8MQvOSliDdNzVl+sMxH4pDcKVnG4eHFJa4cJSZ1fEkRQCxzxrxEgoo9UI5DWeCKzEvmlDJWq2ZDSTmFEFFmcqEil3Vy1Ic9kV887SxLhpzPSiwAxnMzk7IzBI882CXIPMaSlBSZNqxDLFWUyqhKUnz7NS0Ds4Zx9BSc/UmrXNaJXFFhQ7qYKCT46TJFZomUudvLN9ep+L2Y3HwrLovqAlm9ZOUVEB/lo62/tEwm4jsOI4lP80ArpIJE1X25/K3PYvKufso9kElz3UnxUFPuTTsAgzjXfdSupOTd1FBA9Y5AVCjBqEDaaGyJ8/A3ciEWhABLaJMm60a1wok9kgOqLo7mTUKeFMOcg1NanqKAakoyJ1FfpgsFgMShlCJ0sbQWTDQ7jmpWIYwsKV9C+AYEWtHQs0igsF+mFQ3TZrJHQIG4YJCBwyrpqVU9RjZJECKqLpcxg1GV8zZfUKWkHl5MOUpt0xG732NhhSgeC79s6FQyhipWsIZg6usdCZHQfHdsIe24ojnVNBfdkJyOxMWUsPX9I5M82hns7eTLrGA6wYgps2HqDTMbqGILEoTYW9jh55v+htiO6qJgvaHiP6ln0XR5Q1k+aZn61lZdrRKzMpHZMSDR5Kjnry1r7DquIamwcyIe+McoqbKkMnhJqrVCAmuKzGsrJwYE67jTQKt8RdapEiewrsAhSSbFhri9lKR+cTvxjDs0jZYyLbhtonpdjj9jhCyAKOuaByk7jXWaJbC6BTojVDCnsAXM/O+P794SrTlbjq4TopBttbyq1zRy9Eo4POI72+KIk7T6XnzX4aCHdE9W+S8eq7/fK/nJ2LzynwS7rbN03g6ZHxqJmjrNuuGivfxFF12ToJT12l+u01Uuu2X6Su+0F9xNRXFhLhcX7QhpcY5mgek16EVK28xV6+5o4CKtjzJF08Qkeh5NbE6nZ+gTbdDpG7FLoerCBh/n/EqZEpdXZ7nDES3TjDTa3QZnB0R2USlUgLbTQplodPjX/CJHD0cXGToqBXcACGFaQmrRoc0U/3Ly4YQsx/Iiek+/QRyjDRbHYUSOC8spqytlRZSd5cSVCJMj0lZV1cGYjIHSB4AJCeBQ74t7QeIbitX/lDNx8nQ85U7Y67+9+J+P8Q8v3r49+ZEouD47iG3XMY6xGdqYk6e5VVu1sFeU55urRbEpidazg5uz/PW7d+/jt+9+PPlI4ALSleingmwVF38jxVXm8oW+yhftg9eUQ74kGObuo/JufqQks7v7s8oRn4vqZFuGfovlk+nFfKnLEkuq9DYBdKzZkL+hsKVai+VS0QHyuxHIbBbxYq0WFzrQ1QIirmU/3n4w9o5smIIV+FhcGOZs9IoY8y25RWSApsfHZqPrR610OBmyrUQEqrpKTVuxF6JMEDo9H8pm60pRYsdcIi2h+poI7eehH6+oPt9yHoqmnNrPD08t2TPSFu4HW+pncJOal9sSuciNn7lWEX7z4maGa5pNxc+NJdB2sRESiF8ROWayk53WNnh0em4Jnct2nONk7UIOt0PjRjMwb3xO9hCbXqkdh4pTyki2myZkKsjEKpMoGMAEwcwPUNEg6U7n6OXTdi44d1NZBMW2CLg/7vF8I+tNk/kjS+6OkKuGbybmX8rsIoAwdkK3l6ZjutFnUxrsVhVtKouJEZM72w9gtqVC+XIgeRKS8g1aC+Y2ImzESGSOwYYyT3JHKOYAfMPEGkGEs+HY6MkikklicYT7kzvZePMiXWYpxA2cp0eT84GFnfCipkyQdQ+s9YvE7G42/kReb3c3JFCSFfPRNt5p+51BDfCSV1hG0vVv4OIuBwnaDhc81vU4MKQBO0tvY95XLeV9PpTpL5zt7O59RyGYDhjavaC+9TzyXQDT5PZdDU8awtgFq12cbPiRLEuVJ4Hr+JhFZwfG5Rn3/qc+v4awcBT8Ic3f2Sj2olm90IhBoemt7CyICjvvjazfUMq4p9PGBz2QOCLMU0gPJJJEC+pW9++yAGMlRuUxygaBedEvBcpTdrCBhycMh73L7xFRiDEO1PWjkXhkgDiLggE+InxU964eoRV7M9xPHKIfLYsB8rWjvQP2TfsGcS5zDd4u1jZCcCrYBQhuPytJRezf+Tf81lSj2aW80kfmXSxVMpQXpi0gKKWFVuTZ1cjhpMqlCzWB0QLTRLKpFJKOK5GpFZXml0ibBM+Moij0ghklZSIpIBQDeJ1yfWmkEHhB83ccHL0obMi1wZfU8F8i8GLp70QvL/RUiDA8WHWIAi0sfwA/0OGA7hDk8LdIOC8cjvaVbCdlory/+dgYUVyUJp/sd9sw3RjZjundZ8OoKEkv/ks42ChZPbsRgcmcLIbT8bmv3APbfwgLOqx3bZ6d296WefTrNmzSJFb23kZpGGYMpa7UWuVcxLa5I6/69+7f7uzmjkp9WaUEJruKOZMNCAQVujGqqX5G/y5XR0xnovSiSrmqECWpKtn3Stg30Twhda+vUbCRuSrM0/yyvt9hmqOt4urWiOpmuq7jeh6oKI7bB1Ty2WQTwCnTXHJQXwIPgSBJLVF+KpfzofJrt2Cz0ku0TNCwgpfl3qRVOvqLdMzL2ntDEAUrDGbz67SOfMtMvDHhNyTKez+C7P/mmrHeYL5j9oYy8aCtxcjT7la5qGRltfp82qaIbiu3zpyeeyWx5RQS8GPRTnni0nXSexpHwyHKCpnoIGAiHkNtYBPcijgAPIpZ6GQEztlR+QSIlhh/guMGdxOPxb5dmIIRQ6fnuEepp7tb05gxKmsraWgiBt4WJAIf/mkH7Rzk3FKKtoaODBYmqE8NVeccLXDHVnLup7OupHP5rOa+4x5a30tw1yJgduJwiA7MdD/KGy1xLc8lEZ0W0cca7ZNX7xzzTJMnMj+Bvfv46s8vXn94M7JVup1K7izm5q57C+z5sv5pJfByOna09KtjPgTTNatInElaKTI+I4yAKN1NsxyNqk6BonJkvvr06s3Jh/jDyYvXo5a6nTRHlwra02+FRRiN0ZsxnR10J/mVibcj73iYUY2uMUdi8lzVDr5mvo/OwmKEQB3gz+5CshgzT5/iMRcM50bx97fENqSqCD2g2CyD5wm/gnHjfm/Ctl16LQmriNY4zo03M3uEB2avxh63LooLfqcgrp08ZtEYHWDnuiyWH6RWbYfI9EC6ltgPr0/G40lHwIO2cRvF9sUQk4k3MTghVlhB2STojnjUmvUe6CFGOb8CvKTP0Ch+Ex9QAf50PB7Pzr/akPlAY7+Pw0P++Q4zx70rdm80aS8j8yj8snA7XiWFMkmTOWxH0QUgbvy6/YFbWD3MJs8OzKE/WGEbMtpzgO2+7TnCL9veyjO+1W22ZKDZWfa4n21suSrPTIFncEcHXRJv3R5iDZKGgAqVpXHsuIJv/ywr7XBEaa02OuDTM3k/KrBXde/98QJFEhnBMvQbXuQl8BKSUxJg3EmeORxD+6uG/JZRCmQrfIogd1vbcbhwz2neKF9+CxNgjT+cMZDTZ7Nz8t/8XpiotV1qwe7x7MCCdpM9bHiKbjDKnqEyxfpb89IHQ+NofPOQd2L7Bx++Iur81shjX87te+WvcmlWCKcd52CpSLz3z27sefBusck9jI2zs86KfOXey+z6bBGkuTllxtn8CAXEUlbCvRkMfSdgkfPLjZ2e9m/ccEsz0uuBsyoPiiX/Llr8cwb8viK5ZwzpYLUKDngVaXES3PaidiSeDIU3m86eu4Ksah3bg0NRu8DkqscoHYMud70rK2WnMi+KLNgjzDwDqLqnuQBX98C1K1yBcmtI4WPNS+EKHuPd4hj1DV6UzdzrAZQ75s1uGyzwTte2z3B0v3uDaQ4e+XdRzqqU57e/wxvx++6ROYk6Eva0n3cCZ2QPi4Di9ng5BQTr2NVWbko4yHP//dn7IqPCIY/ecCDqF8IN5CLqf0xJ0eZi+48n9LsgRm55JDH4BL9XRaQQ5vwQlSbd+Qp7SAJv4GKYdhwHWmXLkZDHk4j0bG5+Fsdj/CT48R2jbkooZdQu9tQScKISTf6c6kS80IMCG27WfGgrOO0oogIsQfF6bCZwRfH8qRGwI5HU5FJWiaWw3vYOt43EFnFgezojY51Nzkf2ejKbeulVh5DDNyjcP4wlDkV9eGi4eii2h4eGsTRqbpm7XSldkojiSi2pxEanuATmuETPl35TwtXwX7qZ04/MyvXOK8qX0JjkiLa3gQWkSSMzvKtEHwPA82KDk0DcmJwRNJRcU0ieySJpA8UW238OPaDRKfSg1+LobdzuGfqTyc08IVDbmdl3y7+OAR0fPU6083YmWN70RAJm+GJx951oKt7RoSHBomBGHxl+0W3wnP6wPHh/h4YM9zbQ8DtjRmfEZ7jQqCwug2nIPWEciA6agAUCHwqRhLsT+Pl8d7zXY2FERAAw8c+cEe5g99osWempBkvwHuoxEmscT3/aV5OPHCDIEqr0MzmUz2jeXokFEnDSDnzrZNCwY7YGhJidKEQDHJwgnXLfA4U97WBHYR2dOfMS9M561zhmC1n2rPd0PUIudn6L7Y52p7MPWd823TvD1OnlrzOj9sGv3nNjAcEyMIwkOagaQm0HjjAQUmUQQFHWvaVsJ721W6zFRDcYInB7sLYMa0362AEyikCQQEynpYA/qD9mjaeehoQ7tHH579fGRNYyyGOqUvJ4jD/kEjQHrSGBuG8jWrm3hzzI429kjmOomr/kMKem2Oa3li9mBY5YGYTko1Zu+fHKIeS/dul23C7L0lyXqCKIyIkl1l8A54GO3HmLNF20i6lODE7N5a+qKnSMAwTBdkws3Y5J9yYWX+Mv0pQeIAeIyhQuxmV7pjE8TBNVno+xrbsJmw8SZi7nzkI8Out5yDSOxOBMgr0z8byvErf7Fu9bGU7mUP9bh2ROTlI5qdpPPFpB4tnpuEsL+a19d6rY2ukh5dYEMep/V+kc3qEM7anY3ictVHMaNFR//bOhAkyz6zFn/5mamxkfJDSHQgWfwPEmmrwRp3ZoQ6p6pC243kFeHLfvf0rjvdWt8J0Af2BhUtNM5RgLkUU+7dPRJgc8XVAiU8tswNrCFhmo+zxqq12cXGZ8SPiw2NS1DMBdpgt3Nd/t8DLpRgnwiYXxqZ/DPpHXQHbjPsoDsTyv982I+YaLCAsGF+8epiSSPc0pNjjpZvZhvhHh1YNfV/3RkW+PcztGnAYT7PH9jyf8MS/3B4Iphl699EaeYOQHf2TMPXuwzg367Tbz8RJRdpqet59bGCr7I0N7vuV7p27nforORfyse8GRkU1wns65Mv8Z2z/DbtGLiM94wd0xse/f+uGUINy9uBlcex+c88UtBCPmkxrfj3jjMvZcTnBXEjSOJqE50mhLIgurp347abd5bf+F5HsI8K6E0QUqsgbeLV5LHcM5xy4t90VvCrQTrNiv0G4rq/zs/z5VVFtJmQzj2EAYeH45XGkhYATQ+jurq/tUWJ5PsAgP3fc12zI48og8FDaUIoP3a4WwXdKPt10B0VKEYEE74ozlyZgU/Rk5jedjk68cT25RLiOMcfQkNEGNkiLcjZzQF7Vwsn2gm+68Vc9R9XzUvr/G55CfuaMbUA75nSCvOJmyB+oT03pufGfnFvCXjt73beO7jsv3/p0dBPwmkoMjulmummyrSwS7y3W6QBc0azb8pUcd7ttDV37G5uPBW33fOPrDSBxNiNliGk1gas4D+KJ8ClE+pcfPnCind/uJTorTZ0aMuxXxw7yD/UoE4G6H1mMBMo0UX7UQOIogceks7LfyYgpeTOnxE8cL19UzMe5OblhmtMThNKfL0myqQDB2Eytvwcxi8V7ooEw03wop0e0S3xMmnIaZEPqnLsMxOHA0gr2Gn2RZhkecp9lzNsiMd2jIC+HuBLNXmQ+HeDmf73dktEjRUnLQd7+q7O83Drwv5Pa7NK0qPAPr9rZh8gl/H6RV0QAfLa/Dx0nQkRve8sn5H3fVixLRdZyrGimrr1DGT/pl1tOwr2rk7T/SdnGYmgIT3b1OcyUrpFKT56QdNPKJgAfm0j6cPP/i0zCMkqLBq6UhpX1GU56QYj91Svv0vgY8MSqLDQu74WHbHWyatNKaDEDpWP20Y/UfdlnNYe/JNCbnF/P/Y+Jufj+7P7+52zHI0edPv/R0Eg6w+dmYY177hzn9bCCKP+n7T/s5Iueg+/VYKxSPmV9VGFGFgg/zSdpa7ZVGBfKIprK1R2e4pvBAgo70MQh3wdhoiy79zc3/AvsJBNA=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
